<a href="https://colab.research.google.com/github/arthursusilo4/pagarpadi/blob/main/new_riim_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os, json

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = "/content/drive/MyDrive/pest_prediction_v3/main/new-data_attempt-two"
PREPROCESSED_DIR = os.path.join(PROJECT_ROOT, "preprocessed_v3_final")
SCALER_DIR = os.path.join(PROJECT_ROOT, "scalers_v3_final")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "rev2")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Install missing packages for efficiency testing
!pip install keras-flops -q

Mounted at /content/drive


In [9]:
import os, json, numpy as np, tensorflow as tf, pickle
import importlib

# ── FORCE PYTHON TO RELOAD THE UPDATED MODULES FROM DISK ──
import models_v2
importlib.reload(models_v2)
from models_v2 import build_ds_cmag_std

print("🧪 RUNNING DRY-RUN SANITY CHECK FOR ALL CONFIGS...\n")

with open(os.path.join(SCALER_DIR, "feature_meta.json")) as f:
    meta = json.load(f)

SEQ_LEN = meta["seq_len"]; N_FEATURES = meta["n_features"]
K_NEIGHBORS = meta["k_neighbors"]; HORIZONS = meta["horizons"]
N_PESTS = len(meta["pest_cols"]); CLASS_WEIGHTS = meta["class_weights"]

CONFIGS = [
    {"name": "full_focal_hurdle", "variant": "full", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "full_zinb", "variant": "full", "risk_loss": "focal", "intensity_loss": "zinb"},
    {"name": "abl_single_stream", "variant": "single_stream", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "abl_dual_no_attn", "variant": "dual_no_attn", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "abl_std_attn", "variant": "dual_std_attn", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "abl_no_gate", "variant": "no_gate", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "abl_no_mtl", "variant": "no_mtl", "risk_loss": "focal", "intensity_loss": "mse"},
    {"name": "abl_no_diffusion", "variant": "no_diffusion", "risk_loss": "focal", "intensity_loss": "hurdle"},
]

B = 4
dummy_X_clim = np.random.rand(B, SEQ_LEN, N_FEATURES).astype(np.float32)
dummy_X_spat = np.random.rand(B, SEQ_LEN, K_NEIGHBORS).astype(np.float32)
dummy_A_seq = np.random.rand(B, SEQ_LEN, 1).astype(np.float32)

all_clear = True

for cfg in CONFIGS:
    print(f"Checking Config: {cfg['name']}...")
    try:
        model = build_ds_cmag_std(
            SEQ_LEN, N_FEATURES, K_NEIGHBORS, HORIZONS, N_PESTS,
            class_weights=CLASS_WEIGHTS.get("1", None),
            variant=cfg["variant"], risk_loss=cfg["risk_loss"], intensity_loss=cfg["intensity_loss"]
        )

        preds = model.predict([dummy_X_clim, dummy_X_spat, dummy_A_seq], verbose=0)

        dummy_targets = {}
        for h in HORIZONS:
            if f"risk_output_{h}" in model.output_names:
                dummy_targets[f"risk_output_{h}"] = np.random.randint(0, 2, (B, N_PESTS)).astype(np.float32)
            if f"intensity_output_{h}" in model.output_names:
                dummy_targets[f"intensity_output_{h}"] = np.random.rand(B, N_PESTS).astype(np.float32) * 10
            if f"diffusion_output_{h}" in model.output_names:
                dummy_targets[f"diffusion_output_{h}"] = np.random.rand(B, 1 + K_NEIGHBORS).astype(np.float32) * 5

        loss = model.train_on_batch([dummy_X_clim, dummy_X_spat, dummy_A_seq], dummy_targets)

        dummy_file = os.path.join(RESULTS_DIR, "sanity_check.pkl")
        with open(dummy_file, "wb") as f:
            pickle.dump({"preds": preds, "config": cfg}, f)
        with open(dummy_file, "rb") as f:
            _ = pickle.load(f)
        os.remove(dummy_file)

        print(f"  ✅ PASS - Shapes OK, Loss OK, Save/Load OK\n")
        tf.keras.backend.clear_session()

    except Exception as e:
        all_clear = False
        print(f"  ❌ FAIL - Error encountered: {type(e).__name__}: {e}\n")
        tf.keras.backend.clear_session()

print("=====================================================")
if all_clear:
    print("🎉 ALL CHECKS PASSED. You are safe to run the full training cell!")
else:
    print("⚠️ ERRORS DETECTED. Fix the failing configs above before running full training.")

🧪 RUNNING DRY-RUN SANITY CHECK FOR ALL CONFIGS...

Checking Config: full_focal_hurdle...
  ✅ PASS - Shapes OK, Loss OK, Save/Load OK

Checking Config: full_zinb...
  ✅ PASS - Shapes OK, Loss OK, Save/Load OK

Checking Config: abl_single_stream...
  ✅ PASS - Shapes OK, Loss OK, Save/Load OK

Checking Config: abl_dual_no_attn...
  ✅ PASS - Shapes OK, Loss OK, Save/Load OK

Checking Config: abl_std_attn...
  ✅ PASS - Shapes OK, Loss OK, Save/Load OK

Checking Config: abl_no_gate...
  ✅ PASS - Shapes OK, Loss OK, Save/Load OK

Checking Config: abl_no_mtl...
  ✅ PASS - Shapes OK, Loss OK, Save/Load OK

Checking Config: abl_no_diffusion...
  ✅ PASS - Shapes OK, Loss OK, Save/Load OK

🎉 ALL CHECKS PASSED. You are safe to run the full training cell!



--- Training Baseline: STGCN ---


ValueError: `inputs` not connected to `outputs`

In [2]:
%%writefile losses_v2.py
import tensorflow as tf
import numpy as np

EPS = 1e-7

# ── (a) RISK LOSSES ────────────────────────────────────────────
def focal_loss(gamma=2.0, alpha=0.25, class_weights=None):
    """Multi-label focal loss with optional per-class weighting."""
    cw = None if class_weights is None else tf.constant(class_weights, tf.float32)
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, EPS, 1 - EPS)
        bce = -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        w = alpha_t * tf.pow(1 - p_t, gamma)
        if cw is not None:
            # cw shape is (n_pests,), need to broadcast to (batch, n_pests)
            w = w * cw
        return tf.reduce_mean(tf.reduce_sum(w * bce, axis=-1))
    return loss

def asymmetric_loss(gamma_pos=0, gamma_neg=4, clip=0.05, class_weights=None):
    """ASL — Ben-Baruch et al., 2020. Aggressive negative suppression."""
    cw = None if class_weights is None else tf.constant(class_weights, tf.float32)
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        xs_pos = y_pred
        xs_neg = 1 - y_pred
        if clip > 0:
            xs_neg = tf.clip_by_value(xs_neg - clip, 0.0, 1.0)
        los_pos = y_true * tf.math.log(tf.clip_by_value(xs_pos, EPS, 1.0))
        los_neg = (1 - y_true) * tf.math.log(tf.clip_by_value(xs_neg, EPS, 1.0))
        w = (los_pos * tf.pow(1 - xs_pos, gamma_pos) +
             los_neg * tf.pow(xs_neg, gamma_neg))
        if cw is not None:
            w = w * cw
        return -tf.reduce_mean(tf.reduce_sum(w, axis=-1))
    return loss

def class_balanced_focal_loss(samples_per_class, beta=0.9999, gamma=2.0):
    """CB-Focal — Cui et al., 2019. Effective number of samples reweighting."""
    eff_num = 1.0 - np.power(beta, samples_per_class)
    weights = (1 - beta) / np.array(eff_num)
    weights = weights / np.sum(weights) * len(samples_per_class)
    return focal_loss(gamma=gamma, class_weights=weights.tolist())

# ── (b) INTENSITY LOSSES (zero-inflated) ────────────────────────
def tweedie_loss(p=1.5):
    """Tweedie loss for zero-inflated continuous targets (1<p<2)."""
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.math.softplus(y_pred), EPS, 1e8)
        a = -y_true * tf.pow(y_pred, 1 - p) / (1 - p)
        b = tf.pow(y_pred, 2 - p) / (2 - p)
        return tf.reduce_mean(a + b)
    return loss

def hurdle_mse_loss(lambda_bce=1.0, lambda_mse=0.5):
    """Two-stage Hurdle: BCE on presence + masked MSE on positive intensity."""
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        # FIX: Split the 2*n_pests channels properly
        logit, intensity = tf.split(y_pred, 2, axis=-1)

        present = tf.cast(y_true > 0, tf.float32)
        intensity = tf.math.softplus(intensity)  # ≥ 0

        bce = tf.keras.backend.binary_crossentropy(present, logit)
        mask = present
        sq = tf.square(y_true - intensity) * mask
        n_pos = tf.reduce_sum(mask) + EPS
        mse = tf.reduce_sum(sq) / n_pos
        return lambda_bce * tf.reduce_mean(bce) + lambda_mse * mse
    return loss

def zinb_loss(theta_init=1.0):
    """Zero-Inflated Negative Binomial on log1p(hectares)."""
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        # FIX: Split the 3*n_pests channels properly
        pi_logit, mu_log, log_alpha = tf.split(y_pred, 3, axis=-1)

        pi = tf.sigmoid(pi_logit)
        mu = tf.exp(tf.clip_by_value(mu_log, -10, 10))
        alpha = tf.exp(tf.clip_by_value(log_alpha, -5, 5)) + 1.0  # >1 for NB2

        theta = 1.0 / (alpha + EPS)
        y = y_true
        t1 = tf.math.lgamma(y + theta) - tf.math.lgamma(theta) - tf.math.lgamma(y + 1.0)
        t2 = theta * (tf.math.log(theta + EPS) - tf.math.log(theta + mu + EPS))
        t3 = y * (tf.math.log(mu + EPS) - tf.math.log(theta + mu + EPS))
        nb_log = t1 + t2 + t3

        zero_case = tf.math.log(pi + (1 - pi) * tf.exp(nb_log) + EPS)
        nonZero = tf.math.log(1 - pi + EPS) + nb_log
        log_pmf = tf.where(y > 0, nonZero, zero_case)
        return -tf.reduce_mean(log_pmf)
    return loss

Overwriting losses_v2.py


In [6]:
%%writefile models_v2.py
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from losses_v2 import (focal_loss, asymmetric_loss, class_balanced_focal_loss,
                       tweedie_loss, hurdle_mse_loss, zinb_loss)

def build_ds_cmag_std(
    seq_len, n_features, k_neighbors, horizons, n_pests,
    class_weights=None,
    variant="full",
    risk_loss="focal",
    intensity_loss="hurdle",
    hidden_clim=64, hidden_spat=32, dropout=0.2,
    loss_weights=(1.0, 0.8, 0.5)
):
    climate_in = keras.Input(shape=(seq_len, n_features), name="climate_input")
    spatial_in = keras.Input(shape=(seq_len, k_neighbors), name="spatial_input")
    anomaly_in = keras.Input(shape=(seq_len, 1), name="anomaly_input")

    spatial_used = False
    anomaly_used = False

    if variant == "single_stream":
        x_cl = layers.Bidirectional(layers.LSTM(hidden_clim, return_sequences=True))(climate_in)
        x_cl = layers.LayerNormalization()(x_cl)
        cross = x_cl
    else:
        x_cl = layers.Bidirectional(layers.LSTM(hidden_clim, return_sequences=True))(climate_in)
        x_cl = layers.LayerNormalization()(x_cl)
        x_sp = layers.LSTM(hidden_spat, return_sequences=True)(spatial_in)
        x_sp = layers.LayerNormalization()(x_sp)
        cross = layers.Concatenate()([x_cl, x_sp])
        spatial_used = True

    if variant == "dual_no_attn":
        ctx = layers.Flatten()(cross)
    elif variant in ["dual_std_attn", "no_gate"]:
        score = layers.Dense(1, activation="tanh")(cross)
        attn = layers.Softmax(axis=1)(score)
        ctx = layers.Dot(axes=(1, 1))([attn, cross])
        ctx = layers.Flatten()(ctx)
    else:
        score = layers.Dense(1, activation="tanh")(cross)
        gated = layers.Multiply()([score,
                                   layers.Add()([layers.Lambda(lambda x: x * 0 + 1.0)(score),
                                                 anomaly_in])])
        attn = layers.Softmax(axis=1)(gated)
        ctx = layers.Dot(axes=(1, 1))([attn, cross])
        ctx = layers.Flatten()(ctx)
        anomaly_used = True

    # FIX: Connect unused inputs via dummy 0-weight connections to satisfy Keras
    if not spatial_used:
        dummy_sp = layers.GlobalAveragePooling1D()(spatial_in)
        dummy_sp = layers.Dense(ctx.shape[-1], use_bias=False, trainable=False,
                                kernel_initializer="zeros")(dummy_sp)
        ctx = layers.Add()([ctx, dummy_sp])

    if not anomaly_used:
        dummy_an = layers.GlobalAveragePooling1D()(anomaly_in)
        dummy_an = layers.Dense(ctx.shape[-1], use_bias=False, trainable=False,
                                kernel_initializer="zeros")(dummy_an)
        ctx = layers.Add()([ctx, dummy_an])

    x_what = layers.Dense(64, activation="relu")(ctx)
    x_what = layers.Dropout(dropout)(x_what)
    x_how = layers.Dense(64, activation="relu")(ctx)
    x_how = layers.Dropout(dropout)(x_how)

    outputs = []
    if intensity_loss == "hurdle":
        int_ch = 2 * n_pests
    elif intensity_loss == "zinb":
        int_ch = 3 * n_pests
    else:
        int_ch = n_pests

    heads = ["risk", "intensity", "diffusion"]
    if variant == "no_mtl":
        heads = ["risk"]
    elif variant == "no_diffusion":
        heads = ["risk", "intensity"]

    for h in horizons:
        if "risk" in heads:
            outputs.append(layers.Dense(n_pests, activation="sigmoid",
                                        name=f"risk_output_{h}")(x_what))
        if "intensity" in heads:
            outputs.append(layers.Dense(int_ch, activation="linear",
                                        name=f"intensity_output_{h}")(x_what))
        if "diffusion" in heads:
            outputs.append(layers.Dense(1 + k_neighbors, activation="linear",
                                        name=f"diffusion_output_{h}")(x_how))

    model = keras.Model([climate_in, spatial_in, anomaly_in], outputs)

    losses, lw = {}, {}
    l_r, l_i, l_d = loss_weights
    for h in horizons:
        if "risk" in heads:
            if risk_loss == "bce":
                losses[f"risk_output_{h}"] = "binary_crossentropy"
            elif risk_loss == "focal":
                losses[f"risk_output_{h}"] = focal_loss(gamma=2.0, class_weights=class_weights)
            elif risk_loss == "asl":
                losses[f"risk_output_{h}"] = asymmetric_loss(class_weights=class_weights)
            elif risk_loss == "cb_focal":
                losses[f"risk_output_{h}"] = focal_loss(gamma=2.0, class_weights=class_weights)
            lw[f"risk_output_{h}"] = l_r
        if "intensity" in heads:
            if intensity_loss == "mse":
                losses[f"intensity_output_{h}"] = "mse"
            elif intensity_loss == "hurdle":
                losses[f"intensity_output_{h}"] = hurdle_mse_loss()
            elif intensity_loss == "zinb":
                losses[f"intensity_output_{h}"] = zinb_loss()
            elif intensity_loss == "tweedie":
                losses[f"intensity_output_{h}"] = tweedie_loss(p=1.5)
            lw[f"intensity_output_{h}"] = l_i
        if "diffusion" in heads:
            losses[f"diffusion_output_{h}"] = "mse"
            lw[f"diffusion_output_{h}"] = l_d

    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=losses, loss_weights=lw)
    return model

Overwriting models_v2.py


In [8]:
%%writefile baselines.py
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ────────── STGCN (Yu et al., 2018) ──────────
def build_stgcn(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features))
    x = layers.Conv1D(64, 3, activation="relu", padding="causal")(inp)
    x = layers.Conv1D(64, 3, activation="relu", padding="causal")(x)
    # Graph conv approximated as 1×k_neighbors conv on spatial input
    sp = keras.Input(shape=(seq_len, k_neighbors))
    gs = layers.Conv1D(32, 1, activation="relu")(sp)
    merged = layers.Concatenate()([x, gs])
    x = layers.GRU(64)(merged)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, keras.Input(shape=(seq_len, 1))], outs, name="STGCN")

# ────────── Graph WaveNet (Wu et al., 2019) ──────────
def build_graphwavenet(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features))
    sp = keras.Input(shape=(seq_len, k_neighbors))
    # Adaptive adjacency via learnable embeddings
    e1 = keras.backend.random_normal((k_neighbors, 16))
    e2 = keras.backend.random_normal((k_neighbors, 16))
    A = tf.nn.softmax(tf.nn.relu(tf.matmul(e1, tf.transpose(e2))), axis=-1)
    gconv = layers.Dense(32, activation="relu")(
        tf.einsum("btk,kk->btk", sp, A))
    x = layers.Concatenate()([inp, gconv])
    # Dilated TCN
    for d in [1, 2, 4]:
        x = layers.Conv1D(64, 3, padding="causal", dilation_rate=d, activation="relu")(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, keras.Input(shape=(seq_len, 1))], outs, name="GraphWaveNet")

# ────────── AGCRN (Bai et al., 2020) ──────────
def build_agcrn(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features))
    sp = keras.Input(shape=(seq_len, k_neighbors))
    # Node-specific transform
    node_proj = layers.Dense(32)(sp)
    x = layers.Concatenate()([inp, node_proj])
    x = layers.GRU(64, return_sequences=True)(x)
    x = layers.GRU(32)(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, keras.Input(shape=(seq_len, 1))], outs, name="AGCRN")

# ────────── BDGSTN (Mao et al., 2023) ──────────
def build_bdgstn(seq_len, n_features, k_neighbors, horizons, n_pests):
    """Backbone-based dynamic graph + DLinear decomposition."""
    inp = keras.Input(shape=(seq_len, n_features))
    sp = keras.Input(shape=(seq_len, k_neighbors))
    # Trend (moving average) + seasonal residual
    trend = layers.AveragePooling1D(pool_size=3, strides=1, padding="same")(inp)
    seasonal = layers.Subtract()([inp, trend])
    g_dyn = layers.Dense(32, activation="relu")(sp)
    x = layers.Concatenate()([trend, seasonal, g_dyn])
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, keras.Input(shape=(seq_len, 1))], outs, name="BDGSTN")

# ────────── DGN-AEA (Xu et al., 2023) ──────────
def build_dgn_aea(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features))
    sp = keras.Input(shape=(seq_len, k_neighbors))
    # Message passing with adaptive edge attributes
    edge_attr = layers.Dense(16, activation="relu")(sp)  # [B, T, K, 16]
    msg = layers.Dense(32, activation="relu")(edge_attr)
    agg = layers.Lambda(lambda m: tf.reduce_mean(m, axis=-1))(msg)  # mean over neighbours
    x = layers.Concatenate()([inp, agg])
    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.LSTM(32)(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, keras.Input(shape=(seq_len, 1))], outs, name="DGN_AEA")

# ────────── LSTM-DSTGCRN (Pham et al., 2026) ──────────
def build_lstm_dstgcrn(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features))
    sp = keras.Input(shape=(seq_len, k_neighbors))
    g1 = layers.Dense(32, activation="relu")(sp)
    g2 = layers.Dense(32, activation="relu")(g1)
    x = layers.Concatenate()([inp, g2])
    x = layers.LSTM(64, return_sequences=True)(x)
    # Multi-head attention
    attn = layers.MultiHeadAttention(num_heads=2, key_dim=16)(x, x)
    x = layers.Add()([x, attn])
    x = layers.GlobalAveragePooling1D()(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, keras.Input(shape=(seq_len, 1))], outs, name="LSTM_DSTGCRN")

# ────────── E2-CSTP (Huang et al., 2025) ──────────
# Approximated: causal cross-modal Mamba → use BiLSTM + cross-attention
def build_e2_cstp(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features))
    sp = keras.Input(shape=(seq_len, k_neighbors))
    a_in = keras.Input(shape=(seq_len, 1))
    h1 = layers.Bidirectional(layers.LSTM(48, return_sequences=True))(inp)
    h2 = layers.LSTM(32, return_sequences=True)(sp)
    # Cross-modal attention
    cross = layers.MultiHeadAttention(num_heads=2, key_dim=16)(h1, h2)
    x = layers.Concatenate()([h1, cross])
    x = layers.GlobalAveragePooling1D()(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, a_in], outs, name="E2_CSTP")

BASELINES = {
    "STGCN": build_stgcn,
    "GraphWaveNet": build_graphwavenet,
    "AGCRN": build_agcrn,
    "BDGSTN": build_bdgstn,
    "DGN-AEA": build_dgn_aea,
    "LSTM-DSTGCRN": build_lstm_dstgcrn,
    "E2-CSTP": build_e2_cstp,
}

Writing baselines.py


In [9]:
%%writefile eval_risk.py
import numpy as np
import pandas as pd
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                             average_precision_score, matthews_corrcoef,
                             balanced_accuracy_score, f1_score)

def per_class_threshold_tune(y_true_val, y_prob_val, grid=None):
    """For each class, find threshold maximising F1 on validation only."""
    if grid is None:
        grid = np.arange(0.05, 0.55, 0.05)
    n_classes = y_true_val.shape[1]
    thresholds = []
    for c in range(n_classes):
        pos_count = y_true_val[:, c].sum()
        if pos_count == 0:
            thresholds.append(np.nan)  # excluded
            continue
        best_t, best_f1 = 0.5, 0.0
        for t in grid:
            yhat = (y_prob_val[:, c] >= t).astype(int)
            f1 = f1_score(y_true_val[:, c], yhat, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds.append(best_t)
    return np.array(thresholds)

def evaluate_risk_full(y_true, y_prob, thresholds, pest_names):
    """Returns per-class + aggregate metrics table."""
    n_classes = y_true.shape[1]
    rows = []
    for c in range(n_classes):
        pos_count = int(y_true[:, c].sum())
        if pos_count == 0:
            rows.append({"Pest": pest_names[c], "PosCount": 0,
                         "Threshold": np.nan, "Precision": np.nan,
                         "Recall": np.nan, "F1": np.nan, "AUROC": np.nan,
                         "AUPRC": np.nan, "MCC": np.nan, "BACC": np.nan,
                         "Note": "Excluded (no positives)"})
            continue
        t = thresholds[c]
        if np.isnan(t):
            t = 0.5
        yhat = (y_prob[:, c] >= t).astype(int)
        p, r, f, _ = precision_recall_fscore_support(y_true[:, c], yhat,
                                                     average="binary", zero_division=0)
        try:
            auroc = roc_auc_score(y_true[:, c], y_prob[:, c])
        except ValueError:
            auroc = np.nan
        try:
            auprc = average_precision_score(y_true[:, c], y_prob[:, c])
        except ValueError:
            auprc = np.nan
        mcc = matthews_corrcoef(y_true[:, c], yhat)
        bacc = balanced_accuracy_score(y_true[:, c], yhat)
        rows.append({"Pest": pest_names[c], "PosCount": pos_count,
                     "Threshold": t, "Precision": p, "Recall": r, "F1": f,
                     "AUROC": auroc, "AUPRC": auprc, "MCC": mcc, "BACC": bacc,
                     "Note": ""})
    df = pd.DataFrame(rows)
    # Aggregate over valid classes only
    valid = df[df["PosCount"] > 0]
    agg = {
        "Macro-F1 (valid classes only)": valid["F1"].mean(),
        "Micro-F1": f1_score(y_true, (y_prob >= 0.5).astype(int),
                             average="micro", zero_division=0),
        "Weighted-F1": f1_score(y_true, (y_prob >= 0.5).astype(int),
                                average="weighted", zero_division=0),
        "Macro-AUROC (valid)": valid["AUROC"].mean(),
        "Macro-AUPRC (valid)": valid["AUPRC"].mean(),
        "Macro-MCC (valid)": valid["MCC"].mean(),
        "Macro-BACC (valid)": valid["BACC"].mean(),
    }
    return df, agg

def evaluate_risk_with_two_thresholds(y_true_test, y_prob_test,
                                       y_true_val, y_prob_val, pest_names):
    """Compare τ=0.5 vs per-class tuned τ (tuned on val, applied to test)."""
    tuned_t = per_class_threshold_tune(y_true_val, y_prob_val)
    df_tuned, agg_tuned = evaluate_risk_full(y_true_test, y_prob_test, tuned_t, pest_names)
    df_fixed, agg_fixed = evaluate_risk_full(y_true_test, y_prob_test,
                                             np.full_like(tuned_t, 0.5), pest_names)
    return df_tuned, agg_tuned, df_fixed, agg_fixed, tuned_t

Writing eval_risk.py


In [10]:
%%writefile eval_intensity.py
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

def stratified_regression_metrics(y_true_ha, y_pred_ha, severity_labels=None):
    """
    y_true_ha, y_pred_ha: 1D arrays in native hectares (after expm1).
    severity_labels: optional array of {0,1,2,3} per sample.
        0 = zero outbreak (LT=0)
        1 = light (0 < LT ≤ 10 ha)
        2 = moderate (10 < LT ≤ 100 ha)
        3 = severe (LT > 100 ha)
    Returns a dict with overall + per-severity breakdowns.
    """
    if severity_labels is None:
        severity_labels = np.where(y_true_ha == 0, 0,
                          np.where(y_true_ha <= 10, 1,
                          np.where(y_true_ha <= 100, 2, 3)))

    out = {}
    # Overall
    out["Overall_MAE"] = mean_absolute_error(y_true_ha, y_pred_ha)
    out["Overall_RMSE"] = np.sqrt(mean_squared_error(y_true_ha, y_pred_ha))
    # Normalised RMSE (by range)
    out["nRMSE_range"] = out["Overall_RMSE"] / (y_true_ha.max() - y_true_ha.min() + 1e-8)
    # MASE (mean abs scaled error): scale by in-sample naive forecast
    naive = np.mean(np.abs(np.diff(y_true_ha))) + 1e-8
    out["MASE"] = np.mean(np.abs(y_true_ha - y_pred_ha)) / naive

    for sev, name in [(0,"Zero"),(1,"Light"),(2,"Moderate"),(3,"Severe")]:
        m = severity_labels == sev
        if m.sum() == 0:
            continue
        out[f"{name}_N"] = int(m.sum())
        out[f"{name}_MAE"] = mean_absolute_error(y_true_ha[m], y_pred_ha[m])
        out[f"{name}_RMSE"] = np.sqrt(mean_squared_error(y_true_ha[m], y_pred_ha[m]))
        if sev > 0:
            r, _ = pearsonr(y_true_ha[m], y_pred_ha[m])
            out[f"{name}_Pearson"] = r

    # Also "all positives" (sev>0)
    m_pos = severity_labels > 0
    out["AllPos_N"] = int(m_pos.sum())
    out["AllPos_MAE"] = mean_absolute_error(y_true_ha[m_pos], y_pred_ha[m_pos])
    out["AllPos_RMSE"] = np.sqrt(mean_squared_error(y_true_ha[m_pos], y_pred_ha[m_pos]))
    out["AllPos_Spearman"], _ = spearmanr(y_true_ha[m_pos], y_pred_ha[m_pos])
    return out

def residual_diagnostics(y_true_ha, y_pred_ha, save_dir):
    import matplotlib.pyplot as plt
    import os
    os.makedirs(save_dir, exist_ok=True)
    # Scatter
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    ax[0].scatter(y_true_ha, y_pred_ha, s=4, alpha=0.4)
    lim = max(y_true_ha.max(), y_pred_ha.max())
    ax[0].plot([0, lim], [0, lim], 'r--', lw=1)
    ax[0].set_xlabel("Actual Affected Area (ha)")
    ax[0].set_ylabel("Predicted (ha)")
    ax[0].set_title("(a) Actual vs Predicted")
    # Residuals
    resid = y_pred_ha - y_true_ha
    ax[1].hist(resid, bins=80, color="steelblue", edgecolor="k", linewidth=0.3)
    ax[1].set_yscale("log")
    ax[1].set_xlabel("Residual (Pred − Actual)")
    ax[1].set_ylabel("Count (log)")
    ax[1].set_title("(b) Residual Distribution")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "Figure_RegressionDiagnostics.pdf"), bbox_inches="tight")
    plt.savefig(os.path.join(save_dir, "Figure_RegressionDiagnostics.png"), bbox_inches="tight", dpi=300)
    plt.close()

Writing eval_intensity.py


In [11]:
%%writefile covariate_shift.py
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
from sklearn.metrics.pairwise import rbf_kernel

def population_stability_index(expected, actual, bins=10):
    """PSI. >0.25 = major shift; 0.1-0.25 = minor; <0.1 = stable."""
    eps = 1e-4
    edges = np.histogram(np.concatenate([expected, actual]), bins=bins)[1]
    e_pct = np.clip(np.histogram(expected, edges)[0] / len(expected), eps, None)
    a_pct = np.clip(np.histogram(actual, edges)[0] / len(actual), eps, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

def maximum_mean_discrepancy(X, Y, gamma=None):
    """MMD² with RBF kernel."""
    if gamma is None:
        gamma = 1.0 / X.shape[1]
    Kxx = rbf_kernel(X, X, gamma)
    Kyy = rbf_kernel(Y, Y, gamma)
    Kxy = rbf_kernel(X, Y, gamma)
    return float(Kxx.mean() + Kyy.mean() - 2 * Kxy.mean())

def shift_report(df_train, df_val, df_test, feature_cols):
    rows = []
    for c in feature_cols:
        x_tr, x_va, x_te = df_train[c].values, df_val[c].values, df_test[c].values
        rows.append({
            "Feature": c,
            "PSI(train→val)": population_stability_index(x_tr, x_va),
            "PSI(train→test)": population_stability_index(x_tr, x_te),
            "Wasserstein(train→test)": wasserstein_distance(x_tr, x_te),
            "KS_p(train→test)": float(__ks_p(x_tr, x_te)),
        })
    # Multivariate MMD
    mmd_v = maximum_mean_discrepancy(df_train[feature_cols].values, df_val[feature_cols].values)
    mmd_t = maximum_mean_discrepancy(df_train[feature_cols].values, df_test[feature_cols].values)
    return pd.DataFrame(rows), {"MMD(train→val)": mmd_v, "MMD(train→test)": mmd_t}

def __ks_p(a, b):
    from scipy.stats import ks_2samp
    return ks_2samp(a, b).pvalue

def prevalence_drift(df_train, df_test, target_col="LKSJ"):
    p_tr = (df_train[target_col] > 0).mean()
    p_te = (df_test[target_col] > 0).mean()
    return {"train_prevalence": p_tr, "test_prevalence": p_te,
            "delta_pp": (p_te - p_tr) * 100}

Writing covariate_shift.py


In [12]:
%%writefile cv_protocols.py
import pandas as pd
import numpy as np

def expanding_window_walk_forward(df, n_folds=4, min_train_months=42,
                                   val_months=3, test_months=3):
    """Yields (train_dates, val_dates, test_dates) tuples for rolling-origin CV."""
    dates = sorted(df["Date"].unique())
    N = len(dates)
    folds = []
    # Total months per fold
    span = N - min_train_months
    step = max((span - val_months - test_months) // max(n_folds - 1, 1), 1)
    for i in range(n_folds):
        train_end = min_train_months + i * step
        val_end = train_end + val_months
        test_end = val_end + test_months
        if test_end > N:
            break
        folds.append({
            "fold": i + 1,
            "train": dates[:train_end],
            "val":   dates[train_end:val_end],
            "test":  dates[val_end:test_end],
        })
    return folds

def leave_one_kecamatan_out(df):
    """Yields (held_kec, train_df, test_df) tuples for spatial CV."""
    for kec in df["Kecamatan"].unique():
        tr = df[df["Kecamatan"] != kec]
        te = df[df["Kecamatan"] == kec]
        yield kec, tr, te

Writing cv_protocols.py


In [13]:
%%writefile stats_tests.py
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def bootstrap_ci(scores, n_boot=10000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    means = [rng.choice(scores, size=len(scores), replace=True).mean()
             for _ in range(n_boot)]
    lo, hi = np.percentile(means, [(1-ci)/2*100, (1+ci)/2*100])
    return float(np.mean(scores)), float(lo), float(hi)

def cliff_delta(x, y):
    """Cliff's delta effect size for paired non-parametric comparison."""
    n = len(x)
    d = 0
    for i in range(n):
        d += np.sign(y[i] - x[i])
    return d / n

def holm_wilcoxon(per_method_scores, baseline_name, alpha=0.05):
    """
    per_method_scores: dict {method_name: np.array of per-sample scores (paired)}
    Returns DataFrame with p, p_holm, Cliff's δ, 95% CI of difference, significance.
    """
    methods = list(per_method_scores.keys())
    rows = []
    base = per_method_scores[baseline_name]
    for m in methods:
        if m == baseline_name:
            rows.append({"method": m, "p_raw": 1.0, "p_holm": 1.0,
                         "cliff_delta": 0.0, "ci_lo": 0, "ci_hi": 0,
                         "significant": False})
            continue
        diff = per_method_scores[m] - base
        try:
            _, p = wilcoxon(diff)
        except ValueError:
            p = 1.0
        m_diff, lo, hi = bootstrap_ci(diff)
        rows.append({"method": m, "p_raw": p, "p_holm": np.nan,
                     "cliff_delta": cliff_delta(base, per_method_scores[m]),
                     "ci_lo": lo, "ci_hi": hi, "significant": False})
    pvals = [r["p_raw"] for r in rows]
    _, pvals_holm, _, _ = multipletests(pvals, alpha=alpha, method="holm")
    for r, p in zip(rows, pvals_holm):
        r["p_holm"] = p
        r["significant"] = bool(p < alpha)
    return pd.DataFrame(rows)

Writing stats_tests.py


In [14]:
%%writefile efficiency.py
import time
import numpy as np
import tensorflow as tf
import os

def count_params_and_macs(model, sample_inputs):
    """Returns param count, MAC estimate, inference latency, peak GPU mem."""
    n_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    # MAC estimate via Keras FLOP estimator (approximate)
    try:
        from keras_flops import get_flops
        macs = get_flops(model, batch_size=1) / 2  # MACs ≈ FLOPs/2
    except Exception:
        macs = float("nan")

    # Warm-up
    for _ in range(3):
        _ = model(sample_inputs, training=False)

    # Inference latency
    N = 50
    t0 = time.perf_counter()
    for _ in range(N):
        _ = model(sample_inputs, training=False)
    t1 = time.perf_counter()
    latency_ms = (t1 - t0) / N * 1000

    # Peak GPU memory (if available)
    peak_mem_mb = float("nan")
    try:
        peak_mem_mb = float(tf.config.experimental.get_memory_info("GPU:0")["peak"]) / 1e6
    except Exception:
        pass

    return {
        "Params": n_params,
        "MACs": macs,
        "Latency_ms": latency_ms,
        "PeakMem_MB": peak_mem_mb,
    }

def profile_training_time(model, train_ds, val_ds, epochs=5):
    t0 = time.perf_counter()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=0)
    t1 = time.perf_counter()
    return {"train_time_per_epoch_s": (t1 - t0) / epochs, "history": history.history}

Writing efficiency.py


In [15]:
%%writefile topology.py
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

def build_geographic_knn_graph(coords, k=5):
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm="ball_tree").fit(coords)
    _, idx = nbrs.kneighbors(coords)
    return [list(row[1:]) for row in idx]

def build_pest_correlation_graph(pest_matrix, k=5):
    """Nodes connected if their pest time-series correlate strongly."""
    # pest_matrix: [N_nodes, T] (e.g., aggregated LKSJ)
    corr = np.corrcoef(pest_matrix)
    np.fill_diagonal(corr, -1)
    # Top-k per node
    graph = []
    for i in range(corr.shape[0]):
        top = np.argsort(-corr[i])[:k]
        graph.append(top.tolist())
    return graph

def build_climate_similarity_graph(climate_matrix, k=5):
    """Nodes connected if microclimates are similar (cosine)."""
    from sklearn.metrics.pairwise import cosine_similarity
    sim = cosine_similarity(MinMaxScaler().fit_transform(climate_matrix))
    np.fill_diagonal(sim, -1)
    graph = []
    for i in range(sim.shape[0]):
        top = np.argsort(-sim[i])[:k]
        graph.append(top.tolist())
    return graph

def build_hybrid_graph(coords, pest_matrix, climate_matrix, k=5, weights=(0.4,0.3,0.3)):
    """Weighted fusion of geographic distance, pest corr, climate sim."""
    from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
    d_geo = euclidean_distances(coords)
    s_geo = 1 / (1 + d_geo)
    s_pest = np.corrcoef(pest_matrix)
    s_clim = cosine_similarity(MinMaxScaler().fit_transform(climate_matrix))
    # Normalise each
    def norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-8)
    S = weights[0]*norm(s_geo) + weights[1]*norm(s_pest) + weights[2]*norm(s_clim)
    np.fill_diagonal(S, -1)
    return [np.argsort(-S[i])[:k].tolist() for i in range(S.shape[0])]

Writing topology.py


In [16]:
%%writefile sensivity.py
# Sweep grid — ALL hyperparameter selection done on TRAIN+VAL only, never test.
GRID = {
    "k_neighbors": [3, 5, 7, 10],
    "seq_len":     [3, 6, 9, 12],
    "anomaly_threshold": [0.5, 1.0, 1.5, 2.0],   # std-deviations above mean
    "lambda_r":    [0.5, 1.0, 1.5],
    "lambda_i":    [0.4, 0.8, 1.2],
    "lambda_d":    [0.25, 0.5, 1.0],
    "hidden_clim": [32, 64, 128],
    "hidden_spat": [16, 32, 64],
}

# Strategy: One-At-a-Time (OAT) sweep first to identify dominant axes,
# then a 2-level full factorial on the top-2 axes (~4 runs) for interactions.
# For each configuration, train on TRAIN, select epoch by VAL loss,
# report metrics on VAL (not test). Only final chosen config sees TEST.

Writing sensivity.py


In [ ]:
# Old (overclaimed):  y_diffusion = log1p(SSB + 2*SSP + 3*SSJ) for self + K neighbours
# New (honest):       y_diffusion = log1p(LT) for self + K neighbours  (just area, no migration claim)

# We additionally compute, for DIAGNOSTIC ONLY (not a modelling claim):
#  - Source-sink indicator: a desa is a "source" at month t if its LT > 0 AND
#    at least one neighbour's LT increases at month t+1.
#  - Diffusion speed: average distance (km) between source and newly-affected
#    desa in t+1.
#  - Anisotropy index: variance of bearing angles of new outbreaks relative
#    to source.
# These diagnostics describe WHAT the target measures but DO NOT feed into
# the model — keeping claims about "migration" honest.

In [10]:
import os, json, numpy as np, tensorflow as tf
from models_v2 import build_ds_cmag_std
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

with open(os.path.join(SCALER_DIR, "feature_meta.json")) as f:
    meta = json.load(f)

SEQ_LEN = meta["seq_len"]; N_FEATURES = meta["n_features"]
K_NEIGHBORS = meta["k_neighbors"]; HORIZONS = meta["horizons"]
N_PESTS = len(meta["pest_cols"]); CLASS_WEIGHTS = meta["class_weights"]

# Load data
def load_split(name):
    d = {"X_climate": np.load(os.path.join(PREPROCESSED_DIR, f"X_climate_{name}.npy")),
         "X_spatial": np.load(os.path.join(PREPROCESSED_DIR, f"X_spatial_{name}.npy")),
         "A_seq": np.load(os.path.join(PREPROCESSED_DIR, f"A_seq_{name}.npy"))}
    for h in HORIZONS:
        d[f"y_risk_{h}"] = np.load(os.path.join(PREPROCESSED_DIR, f"y_risk_{name}_t{h}.npy"))
        d[f"y_intensity_{h}"] = np.load(os.path.join(PREPROCESSED_DIR, f"y_intensity_{name}_t{h}.npy"))
        d[f"y_diffusion_{h}"] = np.load(os.path.join(PREPROCESSED_DIR, f"y_diffusion_{name}_t{h}.npy"))
    return d

train_data = load_split("train")
val_data = load_split("val")
test_data = load_split("test")

# Define experiments
CONFIGS = [
    {"name": "full_focal_hurdle", "variant": "full", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "full_zinb", "variant": "full", "risk_loss": "focal", "intensity_loss": "zinb"},
    {"name": "abl_single_stream", "variant": "single_stream", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "abl_dual_no_attn", "variant": "dual_no_attn", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "abl_std_attn", "variant": "dual_std_attn", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "abl_no_gate", "variant": "no_gate", "risk_loss": "focal", "intensity_loss": "hurdle"},
    {"name": "abl_no_mtl", "variant": "no_mtl", "risk_loss": "focal", "intensity_loss": "mse"},
    {"name": "abl_no_diffusion", "variant": "no_diffusion", "risk_loss": "focal", "intensity_loss": "hurdle"},
]

for cfg in CONFIGS:
    print(f"\n--- Training {cfg['name']} ---")
    model = build_ds_cmag_std(
        SEQ_LEN, N_FEATURES, K_NEIGHBORS, HORIZONS, N_PESTS,
        class_weights=CLASS_WEIGHTS.get("1", None), # pass h=1 weights for simplicity
        variant=cfg["variant"], risk_loss=cfg["risk_loss"], intensity_loss=cfg["intensity_loss"]
    )
    cb = [EarlyStopping(patience=10, restore_best_weights=True), ReduceLROnPlateau(factor=0.5, patience=5)]

    # Simpler manual map:
    tr_y, va_y = {}, {}
    for h in HORIZONS:
        if f"risk_output_{h}" in model.output_names:
            tr_y[f"risk_output_{h}"] = train_data[f"y_risk_{h}"]
            va_y[f"risk_output_{h}"] = val_data[f"y_risk_{h}"]
        if f"intensity_output_{h}" in model.output_names:
            tr_y[f"intensity_output_{h}"] = train_data[f"y_intensity_{h}"]
            va_y[f"intensity_output_{h}"] = val_data[f"y_intensity_{h}"]
        if f"diffusion_output_{h}" in model.output_names:
            tr_y[f"diffusion_output_{h}"] = train_data[f"y_diffusion_{h}"]
            va_y[f"diffusion_output_{h}"] = val_data[f"y_diffusion_{h}"]

    model.fit([train_data["X_climate"], train_data["X_spatial"], train_data["A_seq"]], tr_y,
              validation_data=([val_data["X_climate"], val_data["X_spatial"], val_data["A_seq"]], va_y),
              epochs=50, batch_size=64, callbacks=cb, verbose=1)

    # Save predictions
    import pickle
    preds = model.predict([test_data["X_climate"], test_data["X_spatial"], test_data["A_seq"]], verbose=0)

    # FIX: Use pickle to save the list of differently-shaped arrays
    with open(os.path.join(RESULTS_DIR, f"preds_{cfg['name']}.pkl"), "wb") as f:
        pickle.dump({"preds": preds, "config": cfg}, f)

    tf.keras.backend.clear_session() # Prevent OOM


--- Training full_focal_hurdle ---
Epoch 1/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 18s 43ms/step - diffusion_output_1_loss: 0.3609 - diffusion_output_2_loss: 0.3469 - diffusion_output_3_loss: 0.3171 - intensity_output_1_loss: 0.7871 - intensity_output_2_loss: 0.7885 - intensity_output_3_loss: 0.5577 - loss: 25.6626 - risk_output_1_loss: 7.6031 - risk_output_2_loss: 8.9825 - risk_output_3_loss: 6.7632 - val_diffusion_output_1_loss: 0.2487 - val_diffusion_output_2_loss: 0.2431 - val_diffusion_output_3_loss: 0.2276 - val_intensity_output_1_loss: 0.3352 - val_intensity_output_2_loss: 0.3082 - val_intensity_output_3_loss: 0.3085 - val_loss: 5.7529 - val_risk_output_1_loss: 1.5566 - val_risk_output_2_loss: 1.5442 - val_risk_output_3_loss: 1.4879 - learning_rate: 0.0010
Epoch 2/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - diffusion_output_1_loss: 0.2632 - diffusion_output_2_loss: 0.2579 - diffusion_output_3_loss: 0.2535 - intensity_output_1_loss: 0.3736 - intensity_output_2_loss: 0.3625 - intensit

In [22]:
%%writefile baselines.py
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── CUSTOM LAYERS USING KERAS.OPS (KERAS 3 COMPATIBLE) ─────────
class AdaptiveAdjacency(layers.Layer):
    """Graph WaveNet self-adaptive adjacency matrix."""
    def __init__(self, k_neighbors, **kwargs):
        super().__init__(**kwargs)
        self.k_neighbors = k_neighbors

    def build(self, input_shape):
        self.e1 = self.add_weight(name='e1', shape=(self.k_neighbors, 16),
                                  initializer='glorot_uniform', trainable=True)
        self.e2 = self.add_weight(name='e2', shape=(self.k_neighbors, 16),
                                  initializer='glorot_uniform', trainable=True)

    def call(self, inputs):
        A = keras.ops.softmax(keras.ops.relu(keras.ops.matmul(self.e1, keras.ops.transpose(self.e2))), axis=-1)
        return keras.ops.matmul(inputs, A)

# ────────── STGCN (Yu et al., 2018) ──────────
def build_stgcn(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features), name="climate_input")
    sp = keras.Input(shape=(seq_len, k_neighbors), name="spatial_input")
    a_in = keras.Input(shape=(seq_len, 1), name="anomaly_input")

    x = layers.Conv1D(64, 3, activation="relu", padding="causal")(inp)
    x = layers.Conv1D(64, 3, activation="relu", padding="causal")(x)
    gs = layers.Conv1D(32, 1, activation="relu")(sp)
    merged = layers.Concatenate()([x, gs, a_in])
    x = layers.GRU(64)(merged)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, a_in], outs, name="STGCN")

# ────────── Graph WaveNet (Wu et al., 2019) ──────────
def build_graphwavenet(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features), name="climate_input")
    sp = keras.Input(shape=(seq_len, k_neighbors), name="spatial_input")
    a_in = keras.Input(shape=(seq_len, 1), name="anomaly_input")

    gconv = AdaptiveAdjacency(k_neighbors)(sp)
    gconv = layers.Dense(32, activation="relu")(gconv)

    x = layers.Concatenate()([inp, gconv, a_in])
    for d in [1, 2, 4]:
        x = layers.Conv1D(64, 3, padding="causal", dilation_rate=d, activation="relu")(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, a_in], outs, name="GraphWaveNet")

# ────────── AGCRN (Bai et al., 2020) ──────────
def build_agcrn(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features), name="climate_input")
    sp = keras.Input(shape=(seq_len, k_neighbors), name="spatial_input")
    a_in = keras.Input(shape=(seq_len, 1), name="anomaly_input")

    node_proj = layers.Dense(32)(sp)
    x = layers.Concatenate()([inp, node_proj, a_in])
    x = layers.GRU(64, return_sequences=True)(x)
    x = layers.GRU(32)(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, a_in], outs, name="AGCRN")

# ────────── BDGSTN (Mao et al., 2023) ──────────
def build_bdgstn(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features), name="climate_input")
    sp = keras.Input(shape=(seq_len, k_neighbors), name="spatial_input")
    a_in = keras.Input(shape=(seq_len, 1), name="anomaly_input")

    trend = layers.AveragePooling1D(pool_size=3, strides=1, padding="same")(inp)
    seasonal = layers.Subtract()([inp, trend])
    g_dyn = layers.Dense(32, activation="relu")(sp)
    x = layers.Concatenate()([trend, seasonal, g_dyn, a_in])
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, a_in], outs, name="BDGSTN")

# ────────── DGN-AEA (Xu et al., 2023) ──────────
def build_dgn_aea(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features), name="climate_input")
    sp = keras.Input(shape=(seq_len, k_neighbors), name="spatial_input")
    a_in = keras.Input(shape=(seq_len, 1), name="anomaly_input")

    # FIX: Removed MeanAggregator to prevent 2D shape collapse. Keep tensors 3D.
    edge_attr = layers.Dense(16, activation="relu")(sp)
    msg = layers.Dense(32, activation="relu")(edge_attr)
    x = layers.Concatenate()([inp, msg, a_in])
    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.LSTM(32)(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, a_in], outs, name="DGN_AEA")

# ────────── LSTM-DSTGCRN (Pham et al., 2026) ──────────
def build_lstm_dstgcrn(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features), name="climate_input")
    sp = keras.Input(shape=(seq_len, k_neighbors), name="spatial_input")
    a_in = keras.Input(shape=(seq_len, 1), name="anomaly_input")

    g1 = layers.Dense(32, activation="relu")(sp)
    g2 = layers.Dense(32, activation="relu")(g1)
    x = layers.Concatenate()([inp, g2, a_in])
    x = layers.LSTM(64, return_sequences=True)(x)
    attn = layers.MultiHeadAttention(num_heads=2, key_dim=16)(x, x)
    x = layers.Add()([x, attn])
    x = layers.GlobalAveragePooling1D()(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, a_in], outs, name="LSTM_DSTGCRN")

# ────────── E2-CSTP (Huang et al., 2025) ──────────
def build_e2_cstp(seq_len, n_features, k_neighbors, horizons, n_pests):
    inp = keras.Input(shape=(seq_len, n_features), name="climate_input")
    sp = keras.Input(shape=(seq_len, k_neighbors), name="spatial_input")
    a_in = keras.Input(shape=(seq_len, 1), name="anomaly_input")

    h1 = layers.Bidirectional(layers.LSTM(48, return_sequences=True))(inp)
    h2 = layers.LSTM(32, return_sequences=True)(sp)
    h1 = layers.Concatenate()([h1, a_in])
    cross = layers.MultiHeadAttention(num_heads=2, key_dim=16)(h1, h2)
    x = layers.Concatenate()([h1, cross])
    x = layers.GlobalAveragePooling1D()(x)
    outs = []
    for h in horizons:
        outs.append(layers.Dense(n_pests, activation="sigmoid", name=f"risk_output_{h}")(x))
        outs.append(layers.Dense(n_pests, name=f"intensity_output_{h}")(x))
        outs.append(layers.Dense(1 + k_neighbors, name=f"diffusion_output_{h}")(x))
    return keras.Model([inp, sp, a_in], outs, name="E2_CSTP")

BASELINES = {
    "STGCN": build_stgcn,
    "GraphWaveNet": build_graphwavenet,
    "AGCRN": build_agcrn,
    "BDGSTN": build_bdgstn,
    "DGN-AEA": build_dgn_aea,
    "LSTM-DSTGCRN": build_lstm_dstgcrn,
    "E2-CSTP": build_e2_cstp,
}

Overwriting baselines.py


In [23]:
import os, numpy as np, tensorflow as tf, pickle
import importlib
import baselines
importlib.reload(baselines)  # Ensure the fixed baselines.py is loaded
from baselines import BASELINES
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

for name, builder in BASELINES.items():
    save_path = os.path.join(RESULTS_DIR, f"preds_{name}.pkl")

    # ── SKIP IF ALREADY TRAINED ──
    if os.path.exists(save_path):
        print(f"\n--- Skipping {name} (Already trained and saved) ---")
        continue

    print(f"\n--- Training Baseline: {name} ---")
    model = builder(SEQ_LEN, N_FEATURES, K_NEIGHBORS, HORIZONS, N_PESTS)

    losses, lw = {}, {}
    for h in HORIZONS:
        losses[f"risk_output_{h}"] = "binary_crossentropy"
        losses[f"intensity_output_{h}"] = "mse"
        losses[f"diffusion_output_{h}"] = "mse"
        lw[f"risk_output_{h}"] = 1.0; lw[f"intensity_output_{h}"] = 0.8; lw[f"diffusion_output_{h}"] = 0.5

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=losses, loss_weights=lw)

    tr_y, va_y = {}, {}
    for h in HORIZONS:
        tr_y[f"risk_output_{h}"] = train_data[f"y_risk_{h}"]; va_y[f"risk_output_{h}"] = val_data[f"y_risk_{h}"]
        tr_y[f"intensity_output_{h}"] = train_data[f"y_intensity_{h}"]; va_y[f"intensity_output_{h}"] = val_data[f"y_intensity_{h}"]
        tr_y[f"diffusion_output_{h}"] = train_data[f"y_diffusion_{h}"]; va_y[f"diffusion_output_{h}"] = val_data[f"y_diffusion_{h}"]

    cb = [EarlyStopping(patience=10, restore_best_weights=True), ReduceLROnPlateau(factor=0.5, patience=5)]
    model.fit([train_data["X_climate"], train_data["X_spatial"], train_data["A_seq"]], tr_y,
              validation_data=([val_data["X_climate"], val_data["X_spatial"], val_data["A_seq"]], va_y),
              epochs=50, batch_size=64, callbacks=cb, verbose=1)

    preds = model.predict([test_data["X_climate"], test_data["X_spatial"], test_data["A_seq"]], verbose=0)

    with open(save_path, "wb") as f:
        pickle.dump({"preds": preds}, f)

    tf.keras.backend.clear_session()

print("\n✅ All baselines processed!")


--- Skipping STGCN (Already trained and saved) ---

--- Skipping GraphWaveNet (Already trained and saved) ---

--- Skipping AGCRN (Already trained and saved) ---

--- Skipping BDGSTN (Already trained and saved) ---

--- Training Baseline: DGN-AEA ---
Epoch 1/50
131/131 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - diffusion_output_1_loss: 0.2797 - diffusion_output_2_loss: 0.2697 - diffusion_output_3_loss: 0.2617 - intensity_output_1_loss: 0.0129 - intensity_output_2_loss: 0.0168 - intensity_output_3_loss: 0.0106 - loss: 1.2857 - risk_output_1_loss: 0.2738 - risk_output_2_loss: 0.2929 - risk_output_3_loss: 0.2761 - val_diffusion_output_1_loss: 0.2514 - val_diffusion_output_2_loss: 0.2448 - val_diffusion_output_3_loss: 0.2306 - val_intensity_output_1_loss: 0.0056 - val_intensity_output_2_loss: 0.0053 - val_intensity_output_3_loss: 0.0050 - val_loss: 0.6310 - val_risk_output_1_loss: 0.0847 - val_risk_output_2_loss: 0.0883 - val_risk_output_3_loss: 0.0825 - learning_rate: 0.0010
Epoch 2/50
131/131

In [24]:
import numpy as np, pandas as pd, os, json, pickle
from eval_risk import evaluate_risk_with_two_thresholds
from eval_intensity import stratified_regression_metrics

PEST_NAMES = [p.replace("LKSJ_", "") for p in meta["pest_cols"]]

# Load true test labels
y_true_risk = {h: test_data[f"y_risk_{h}"] for h in HORIZONS}
y_true_int = {h: np.expm1(test_data[f"y_intensity_{h}"]) for h in HORIZONS}
y_true_dif = {h: np.expm1(test_data[f"y_diffusion_{h}"]) for h in HORIZONS}

# True val labels for threshold tuning
y_val_risk = {h: val_data[f"y_risk_{h}"] for h in HORIZONS}

eval_results = []

for file in os.listdir(RESULTS_DIR):
    if not file.startswith("preds_") or not file.endswith(".pkl"): continue
    model_name = file.replace("preds_", "").replace(".pkl", "")

    with open(os.path.join(RESULTS_DIR, file), "rb") as f:
        data = pickle.load(f)
    preds = data["preds"]

    # Determine output structure based on number of predictions
    n_outputs = len(preds)
    has_intensity = True
    has_diffusion = True
    if n_outputs == 3: # no_mtl variant
        has_intensity = False; has_diffusion = False
    elif n_outputs == 6: # no_diffusion variant
        has_diffusion = False

    res = {"Model": model_name}
    idx = 0
    for h in HORIZONS:
        # 1. Risk (always present)
        pred_risk = preds[idx]
        idx += 1

        # For proper threshold tuning, we should use val_preds. Since we didn't save them,
        # we will use a fixed threshold of 0.5 for this quick evaluation table.
        # We pass test data for both to avoid crashing, effectively evaluating at 0.5.
        try:
            _, agg_tuned, _, _, _ = evaluate_risk_with_two_thresholds(
                y_true_risk[h], pred_risk, y_val_risk[h], pred_risk, PEST_NAMES)
            res[f"F1_t{h}"] = agg_tuned["Macro-F1 (valid classes only)"]
            res[f"AUROC_t{h}"] = agg_tuned["Macro-AUROC (valid)"]
        except Exception as e:
            print(f"  Risk eval error for {model_name} t{h}: {e}")
            res[f"F1_t{h}"] = np.nan
            res[f"AUROC_t{h}"] = np.nan

        # 2. Intensity (optional)
        if has_intensity:
            pred_int_raw = preds[idx]
            idx += 1

            # Handle hurdle/zinb splits
            if pred_int_raw.shape[-1] == 2 * len(PEST_NAMES):
                _, pred_int_raw = np.split(pred_int_raw, 2, axis=-1)
            elif pred_int_raw.shape[-1] == 3 * len(PEST_NAMES):
                _, pred_int_raw, _ = np.split(pred_int_raw, 3, axis=-1)

            pred_int_ha = np.expm1(pred_int_raw)
            try:
                int_metrics = stratified_regression_metrics(y_true_int[h].flatten(), pred_int_ha.flatten())
                res[f"RMSE_pos_t{h}"] = int_metrics["AllPos_RMSE"]
            except Exception as e:
                print(f"  Intensity eval error for {model_name} t{h}: {e}")
                res[f"RMSE_pos_t{h}"] = np.nan
        else:
            res[f"RMSE_pos_t{h}"] = np.nan

        # 3. Diffusion (optional)
        if has_diffusion:
            pred_dif = preds[idx]
            idx += 1
            # You can add diffusion metrics here if needed
            res[f"Dif_RMSE_t{h}"] = np.nan # Placeholder for now

    eval_results.append(res)

df_results = pd.DataFrame(eval_results).set_index("Model")
df_results.to_csv(os.path.join(RESULTS_DIR, "summary_results.csv"))
print("\n✅ Evaluation Complete! Results saved to summary_results.csv")
print(df_results)


✅ Evaluation Complete! Results saved to summary_results.csv
                      F1_t1  AUROC_t1  RMSE_pos_t1  Dif_RMSE_t1     F1_t2  \
Model                                                                       
full_focal_hurdle  0.101438  0.641635     1.288117          NaN  0.079446   
full_zinb          0.106771  0.621983     1.767861          NaN  0.070696   
abl_single_stream  0.081479  0.622711     1.264058          NaN  0.068808   
abl_dual_no_attn   0.088850  0.603329     1.170897          NaN  0.067055   
abl_std_attn       0.098852  0.646581     1.249657          NaN  0.076258   
abl_no_gate        0.108456  0.641948     1.272936          NaN  0.080008   
abl_no_mtl         0.112939  0.679281          NaN          NaN  0.075304   
abl_no_diffusion   0.096381  0.623431     1.251173          NaN  0.081275   
STGCN              0.105057  0.723848     0.912644          NaN  0.065668   
GraphWaveNet       0.089524  0.696775     0.910430          NaN  0.072662   
AGCRN          

In [25]:
from efficiency import count_params_and_macs
from models_v2 import build_ds_cmag_std
from baselines import BASELINES
import pandas as pd, numpy as np

# Dummy sample inputs for profiling
sample = [test_data["X_climate"][:1], test_data["X_spatial"][:1], test_data["A_seq"][:1]]

eff_rows = []
# Profile main model
m = build_ds_cmag_std(SEQ_LEN, N_FEATURES, K_NEIGHBORS, HORIZONS, N_PESTS, variant="full")
eff_rows.append({"Model": "DS-CMAG-STD", **count_params_and_macs(m, sample)})

# Profile baselines
for name, builder in BASELINES.items():
    m = builder(SEQ_LEN, N_FEATURES, K_NEIGHBORS, HORIZONS, N_PESTS)
    eff_rows.append({"Model": name, **count_params_and_macs(m, sample)})

df_eff = pd.DataFrame(eff_rows)
df_eff.to_csv(os.path.join(RESULTS_DIR, "efficiency_results.csv"), index=False)
print(df_eff)

          Model  Params  MACs  Latency_ms  PeakMem_MB
0   DS-CMAG-STD   84537   NaN  173.300514         NaN
1         STGCN   57878   NaN   64.984994         NaN
2  GraphWaveNet   70070   NaN   29.407076         NaN
3         AGCRN   37782   NaN   96.923158         NaN
4        BDGSTN   39574   NaN   23.463185         NaN
5       DGN-AEA   48726   NaN   94.186810         NaN
6  LSTM-DSTGCRN   50070   NaN   56.372367         NaN
7       E2-CSTP   69827   NaN  128.384399         NaN


In [26]:
import os, numpy as np, pandas as pd, pickle

# Load true test labels
y_true_dif = {h: np.expm1(test_data[f"y_diffusion_{h}"]) for h in HORIZONS}

spatial_results = []

for file in os.listdir(RESULTS_DIR):
    if not file.startswith("preds_") or not file.endswith(".pkl"): continue
    model_name = file.replace("preds_", "").replace(".pkl", "")

    with open(os.path.join(RESULTS_DIR, file), "rb") as f:
        data = pickle.load(f)
    preds = data["preds"]

    n_outputs = len(preds)
    has_diffusion = True
    if n_outputs == 3: has_diffusion = False
    elif n_outputs == 6: has_diffusion = False

    res = {"Model": model_name}
    idx = 0
    for h in HORIZONS:
        # Risk is index 0
        risk_pred = preds[idx]; idx += 1

        # Intensity is index 1
        if n_outputs != 3: # if it has intensity
            int_pred = preds[idx]; idx += 1

        # Diffusion is index 2
        if has_diffusion:
            dif_pred = preds[idx]; idx += 1
            from scipy.stats import pearsonr
            # Flatten and calculate Pearson
            r, _ = pearsonr(y_true_dif[h].flatten(), dif_pred.flatten())
            res[f"Spatr_t{h}"] = r
        else:
            res[f"Spatr_t{h}"] = np.nan

    spatial_results.append(res)

df_spatial = pd.DataFrame(spatial_results).set_index("Model")
print("\n=== Spatial Diffusion Pearson r ===")
print(df_spatial)

# Merge with previous results
df_main = pd.read_csv(os.path.join(RESULTS_DIR, "summary_results.csv")).set_index("Model")
df_final = df_main.join(df_spatial)
df_final.to_csv(os.path.join(RESULTS_DIR, "final_summary.csv"))
print("\n✅ Saved to final_summary.csv")


=== Spatial Diffusion Pearson r ===
                   Spatr_t1  Spatr_t2  Spatr_t3
Model                                          
full_focal_hurdle  0.278889  0.163087  0.117510
full_zinb          0.401375  0.204080  0.136153
abl_single_stream  0.109276  0.082740  0.060896
abl_dual_no_attn   0.215861  0.144451  0.101810
abl_std_attn       0.349367  0.197713  0.136686
abl_no_gate        0.298260  0.183986  0.131051
abl_no_mtl              NaN       NaN       NaN
abl_no_diffusion        NaN       NaN       NaN
STGCN              0.533301  0.233230  0.144151
GraphWaveNet       0.360398  0.155559  0.088766
AGCRN              0.543238  0.247745  0.144484
BDGSTN             0.526407  0.243745  0.164087
DGN-AEA            0.003958  0.015132  0.012734
LSTM-DSTGCRN       0.532242  0.234008  0.146892
E2-CSTP            0.492755  0.228217  0.149109

✅ Saved to final_summary.csv
